# Tahap 3: Case Retrieval
**SubCPMK-3 — Penalaran Komputer | NIM: 202310370311358**

Tujuan: Membangun sistem retrieval menggunakan **TF-IDF + SVM** untuk menemukan kasus putusan yang paling mirip dengan query kasus baru.

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

PROCESSED_DIR = Path('../data/processed')
EVAL_DIR      = Path('../data/eval')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'cases_clean.csv')
print(f'Dataset: {len(df)} kasus')
print(f'Label distribution:\n{df["label"].value_counts()}')

## 3.1 Representasi Vektor: TF-IDF

In [ ]:
# Hapus baris dengan label 'unknown' untuk training
df_model = df[df['label'] != 'unknown'].copy()
print(f'Kasus untuk training: {len(df_model)}')

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),    # Unigram + bigram
    min_df=2,              # Minimal muncul di 2 dokumen
    sublinear_tf=True,     # Gunakan log TF
)

X = tfidf.fit_transform(df_model['text_clean'].fillna(''))
print(f'Shape matriks TF-IDF: {X.shape}')
print(f'Contoh fitur teratas: {tfidf.get_feature_names_out()[:10]}')

## 3.2 Splitting Data (Train/Test = 80:20)

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df_model['label'])
case_ids = df_model['case_id'].values

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, case_ids,
    test_size=0.2,
    random_state=42,
    stratify=y if len(set(y)) > 1 else None
)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Label classes: {le.classes_}')

## 3.3 Model Retrieval: SVM Classifier

In [ ]:
# Training SVM
svm = SVC(kernel='linear', probability=True, C=1.0, random_state=42)
svm.fit(X_train, y_train)

# Evaluasi awal
train_acc = svm.score(X_train, y_train)
test_acc  = svm.score(X_test, y_test)
print(f'Train Accuracy : {train_acc:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')

## 3.4 Fungsi Retrieval: retrieve(query, k=5)

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS_ID = set(stopwords.words('indonesian'))

# Simpan semua vektor kasus dalam case base
X_all      = tfidf.transform(df_model['text_clean'].fillna(''))
all_ids    = df_model['case_id'].values
all_labels = df_model['label'].values

def preprocess_query(query):
    """Preprocessing query sesuai pipeline training."""
    query = query.lower()
    query = re.sub(r'[^a-zA-Z\s]', ' ', query)
    tokens = [t for t in query.split() if t not in STOPWORDS_ID and len(t) > 2]
    return ' '.join(tokens)

def retrieve(query: str, k: int = 5):
    """
    Fungsi retrieval utama CBR.
    1. Preprocessing query
    2. Hitung vektor TF-IDF query
    3. Hitung cosine similarity dengan semua kasus
    4. Kembalikan top-k case_id
    
    Returns: list of (case_id, similarity_score)
    """
    # Step 1: Preprocess
    q_clean = preprocess_query(query)
    
    # Step 2: Vektorisasi
    q_vec = tfidf.transform([q_clean])
    
    # Step 3: Cosine similarity
    sims = cosine_similarity(q_vec, X_all).flatten()
    
    # Step 4: Top-k
    top_k_idx = np.argsort(sims)[::-1][:k]
    results = [(all_ids[i], float(sims[i])) for i in top_k_idx]
    return results

# Uji fungsi retrieval
test_query = "terdakwa membawa sabu-sabu seberat 5 gram pasal 112 narkotika"
top5 = retrieve(test_query, k=5)
print('Top-5 kasus termirip:')
for rank, (cid, score) in enumerate(top5, 1):
    print(f'  {rank}. {cid} | similarity={score:.4f}')

## 3.5 Pengujian Awal & Buat queries.json

In [ ]:
# Buat 5-10 query uji dari data test
# Ground truth = label kasus dari data test

queries_eval = []
for i in range(min(10, len(ids_test))):
    row = df_model[df_model['case_id'] == ids_test[i]].iloc[0]
    queries_eval.append({
        'query_id'        : f'q{i+1:02d}',
        'query_text'      : row['text_clean'][:300],  # Potongan teks sebagai query
        'ground_truth_id' : row['case_id'],
        'ground_truth_label': row['label'],
    })

with open(EVAL_DIR / 'queries.json', 'w', encoding='utf-8') as f:
    json.dump(queries_eval, f, ensure_ascii=False, indent=2)

print(f'Saved: data/eval/queries.json ({len(queries_eval)} query)')

# Uji setiap query
for q in queries_eval:
    results = retrieve(q['query_text'], k=5)
    top1_id = results[0][0]
    match = '✓' if top1_id == q['ground_truth_id'] else '✗'
    print(f"{match} {q['query_id']}: GT={q['ground_truth_id']} | Top1={top1_id}")

## 3.6 Simpan Model

In [ ]:
MODEL_DIR = Path('../data/processed')

with open(MODEL_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(MODEL_DIR / 'svm_model.pkl', 'wb') as f:
    pickle.dump(svm, f)

with open(MODEL_DIR / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Simpan metadata untuk retrieval
np.save(MODEL_DIR / 'tfidf_matrix.npy', X_all.toarray())
pd.DataFrame({'case_id': all_ids, 'label': all_labels}).to_csv(
    MODEL_DIR / 'case_index.csv', index=False
)

print('Model dan indeks kasus tersimpan.')
print('\n>>> Tahap 3 SELESAI. Lanjut ke 04_solution_reuse.ipynb')